# Plot Checkpoint Probe Predictions

Loads a checkpoint and plots forecast-head predictions from the encoder model. For pretraining-only checkpoints, this forecast head is diagnostic unless it has been fine-tuned.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import torch

CHECKPOINT = Path(r'D:/TradingData/quant-research-workbench/market_data/models/masked_event_model/v1/mem-v1-d512-e2-t8-d4-mask70-chunk500-nov2025/checkpoint_last.pt')
print('checkpoint exists', CHECKPOINT.exists(), CHECKPOINT)

In [ ]:
from pathlib import Path
import json
import numpy as np
import polars as pl
from torch.utils.data import DataLoader
from research.masked_event_model.v1.config import DataConfig, ModelConfig, MaskConfig
from research.masked_event_model.v1.data import EventChunkDataset, discover_chunk_files, target_horizons_from_columns
from research.masked_event_model.v1.masking import build_structured_masks
from research.masked_event_model.v1.model import MaskedEventAutoencoder
from research.masked_event_model.v1.schema import QUOTE_FEATURE_COLUMNS, TRADE_FEATURE_COLUMNS, CHUNK_SUMMARY_COLUMNS
from research.masked_event_model.v1.targets import decode_binary_magnitude_logits_to_bps

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data_config = DataConfig(cache_root=Path(r'D:/market-data/flatfiles/us_stocks_sip/derived/event_chunks_v2'), max_files=8)
files = discover_chunk_files(data_config, start_date=data_config.validation_start_date, end_date=data_config.validation_end_date)
horizon_count = len(target_horizons_from_columns(pl.scan_parquet(str(files[0].path)).collect_schema().names()))
model_config = ModelConfig()
model = MaskedEventAutoencoder(
    quote_feature_count=len(QUOTE_FEATURE_COLUMNS), trade_feature_count=len(TRADE_FEATURE_COLUMNS), summary_feature_count=len(CHUNK_SUMMARY_COLUMNS),
    context_chunks=data_config.context_chunks, max_quote_events=data_config.max_quote_events, max_trade_events=data_config.max_trade_events,
    max_total_events=data_config.max_total_events, horizon_count=horizon_count, target_bit_count=data_config.target_bit_count, config=model_config,
).to(device)
if CHECKPOINT.exists():
    ckpt = torch.load(CHECKPOINT, map_location=device)
    model.load_state_dict(ckpt['model'])
model.eval()
batch = next(iter(DataLoader(EventChunkDataset(config=data_config, split='validation', batch_size=32, seed=19), batch_size=None, num_workers=0)))
moved = {k: (v.to(device) if hasattr(v, 'to') else v) for k, v in batch.items()}
masks = build_structured_masks(quote_values=moved['quote_values'], trade_values=moved['trade_values'], chunk_summary=moved['chunk_summary'], event_kinds=moved['event_kinds'], config=MaskConfig())
with torch.no_grad():
    out = model(moved['quote_values'], moved['trade_values'], moved['event_kinds'], moved['event_indices'], moved['chunk_summary'], masks)
pred = decode_binary_magnitude_logits_to_bps(out.forecast_logits.detach().cpu().numpy()).reshape(batch['target_bps'].shape)
target = batch['target_bps'].numpy()
plt.figure(figsize=(14, 5))
plt.plot(target[:, 0, 0], label='target h1 bps', linewidth=2)
plt.plot(pred[:, 0, 0], label='prediction h1 bps', linewidth=1)
plt.legend(); plt.grid(True); plt.title('Masked Event v1 checkpoint predictions vs target'); plt.show()